In [1]:
!pip install ultralytics opencv-python


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # lightweight + fast

In [3]:
import cv2
import numpy as np

def detect_phone_usage(frame):
    results = model(frame)[0]

    persons = []
    phones = []

    for box in results.boxes:
        cls = int(box.cls[0])
        name = model.names[cls]
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        if name == "person":
            persons.append((x1, y1, x2, y2))
        elif name == "cell phone":
            phones.append((x1, y1, x2, y2))

    using_phone = False

    # Check distance between phone & person (face area approx upper body)
    for (px1, py1, px2, py2) in persons:
        face_y = py1 + (py2 - py1) // 3  # approx face region

        for (x1, y1, x2, y2) in phones:
            phone_center = ((x1 + x2)//2, (y1 + y2)//2)

            if py1 < phone_center[1] < face_y:
                using_phone = True

                # Draw box
                cv2.rectangle(frame, (x1,y1), (x2,y2), (0,0,255), 2)

    return frame, using_phone

In [4]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame, using_phone = detect_phone_usage(frame)

    label = "USING PHONE 🚨" if using_phone else "SAFE ✅"
    color = (0,0,255) if using_phone else (0,255,0)

    cv2.putText(frame, label, (20,40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

    cv2.imshow("Driver Monitor", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


0: 480x640 1 person, 217.6ms
Speed: 10.6ms preprocess, 217.6ms inference, 16.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 241.1ms
Speed: 7.1ms preprocess, 241.1ms inference, 2.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 265.9ms
Speed: 5.9ms preprocess, 265.9ms inference, 2.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 263.4ms
Speed: 3.4ms preprocess, 263.4ms inference, 3.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 271.8ms
Speed: 3.2ms preprocess, 271.8ms inference, 2.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 259.3ms
Speed: 3.9ms preprocess, 259.3ms inference, 2.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 267.8ms
Speed: 3.4ms preprocess, 267.8ms inference, 2.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 274.8ms
Speed: 3.8ms preprocess, 274.8ms inference, 2.9ms postprocess per image 

KeyboardInterrupt: 

In [6]:
import cv2
import base64
import numpy as np
from IPython.display import display, Javascript
from ultralytics import YOLO

In [7]:
def take_photo():
    js = Javascript('''
    async function takePhoto() {
      const video = document.createElement('video');
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      video.srcObject = stream;
      await video.play();

      // Wait for camera to initialize
      await new Promise(resolve => setTimeout(resolve, 1000));

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);

      stream.getTracks().forEach(track => track.stop());

      return canvas.toDataURL('image/jpeg');
    }
    takePhoto();
    ''')

    display(js)
    data = eval_js('takePhoto()')

    binary = base64.b64decode(data.split(',')[1])
    image = np.frombuffer(binary, dtype=np.uint8)
    return cv2.imdecode(image, cv2.IMREAD_COLOR)

In [12]:
pip install pyttsx3


   ------------- -------------------------- 1/3 [comtypes]
   ------------- -------------------------- 1/3 [comtypes]
   ------------- -------------------------- 1/3 [comtypes]
   ------------- -------------------------- 1/3 [comtypes]
   ------------- -------------------------- 1/3 [comtypes]
   ------------- -------------------------- 1/3 [comtypes]
   ------------- -------------------------- 1/3 [comtypes]
   ------------- -------------------------- 1/3 [comtypes]
   ------------- -------------------------- 1/3 [comtypes]
   ------------- -------------------------- 1/3 [comtypes]
   ------------- -------------------------- 1/3 [comtypes]
   -------------------------- ------------- 2/3 [pyttsx3]
   ---------------------------------------- 3/3 [pyttsx3]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import cv2
import time
from ultralytics import YOLO
import pyttsx3

# Init TTS
engine = pyttsx3.init()
engine.setProperty('rate', 160)  # speed

model = YOLO("yolov8n.pt")

cap = cv2.VideoCapture(0)

last_alert_time = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)[0]

    phone_found = False

    for box in results.boxes:
        cls = int(box.cls[0])
        label = model.names[cls]

        if label == "cell phone" and box.conf[0] > 0.5:
            phone_found = True

    # 🔊 Voice alert (non-spam)
    if phone_found and (time.time() - last_alert_time > 5):
        print("🚨 Phone detected!")
        last_alert_time = time.time()

        engine.say("Please do not use phone while driving")
        engine.say("कृपया गाड़ी चलाते समय फोन का उपयोग न करें")
        engine.runAndWait()

    cv2.imshow("frame", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


0: 480x640 2 persons, 411.9ms
Speed: 8.1ms preprocess, 411.9ms inference, 4.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 358.5ms
Speed: 4.8ms preprocess, 358.5ms inference, 2.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 376.7ms
Speed: 4.5ms preprocess, 376.7ms inference, 4.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 315.9ms
Speed: 4.7ms preprocess, 315.9ms inference, 2.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 358.8ms
Speed: 3.5ms preprocess, 358.8ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 363.5ms
Speed: 4.6ms preprocess, 363.5ms inference, 5.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 443.2ms
Speed: 4.7ms preprocess, 443.2ms inference, 3.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 342.9ms
Speed: 6.1ms preprocess, 342.9ms inference, 4.5ms postprocess per ima

KeyboardInterrupt: 

: 